In [ ]:
#import feedparser
from bs4 import BeautifulSoup
import requests
import re
from datetime import datetime



In [11]:
response = requests.get("https://campusbelfortmontbeliard.fr/emissions/au-coeur-de-la-meule/")
soup = BeautifulSoup(response.content, 'html.parser')

#keyword = "acdlm"
keyword_pattern = re.compile(r'\bacdlm\b')
matches = soup.find_all('a', href=keyword_pattern)
slugs=[re.search(pattern=r'replay/(.+)/',string=m['href']).group(1) for m in matches]

In [30]:
def convertir_date_iso_vers_rfc(date_iso):
    """Convertit 2026-03-19T08:01:07+00:00 en RFC 822"""
    dt = datetime.fromisoformat(date_iso.replace('Z', '+00:00'))
    return dt.strftime('%a, %d %b %Y %H:%M:%S %z')


Thu, 19 Mar 2026 08:01:07 +0000


In [31]:
# 1. Parser la page web pour extraire les liens MP3
def extraire_episode(s,base_url='https://campusbelfortmontbeliard.fr/replay/'): #s=slug
    slug_response = requests.get(base_url+s)
    slug_soup = BeautifulSoup(slug_response.content, 'html.parser')
    episode={'title':slug_soup.find('title').text,
    'link':base_url+s,
    'mp3_url':slug_soup.find(class_="section page-body").find('button', attrs={'aria-label':'Player'})['data-file'],
    'guid':s,
    'description':slug_soup.find(class_="titleside-body content").text.strip(),
    'pubDate':convertir_date_iso_vers_rfc(slug_soup.find('meta',attrs={'property':"article:modified_time"})['content']),
    'season':2026,
    'duration':slug_soup.find(class_="section page-body").find(class_="audioplayer-suptitle").text[-8:].strip(),
    'image':slug_soup.find('meta',attrs={"property":"og:image"})['content']}
    temp=re.search(r'ACDLM S(\d+)-Ep(\d+)',episode['title'])
    episode['season']= temp.group(1)
    episode['episode']= temp.group(2)   
    return episode



In [32]:
base_url='https://campusbelfortmontbeliard.fr/replay/'
episodes =[extraire_episode(s) for s in slugs]

In [ ]:
# 2. Générer le flux RSS
def generer_rss(episodes):
    from datetime import datetime
    
    rss = f'''<?xml version="1.0" encoding="UTF-8"?>
<rss version="2.0" xmlns:itunes="http://www.itunes.com/dtds/podcast-1.0.dtd">
  <channel>
    <title>Au cœur de la Meule</title>
    <link>https://campusbelfortmontbeliard.fr/emissions/au-coeur-de-la-meule/</link>
    <description>Podcast Radio Campus Belfort-Montbéliard</description>
    <itunes:image href="https://campusbelfortmontbeliard.fr/wp-content/uploads/sites/2/2026/01/visuel-au-coeur-de-la-meule-400x400.jpg"/>
    <itunes:author>Radio Campus Belfort-Montbéliard</itunes:author>
    <itunes:owner>
  <itunes:name>Radio Campus Belfort-Montbéliard</itunes:name>
  <itunes:email>acdlm88.8@proton.me</itunes:email>
</itunes:owner>
<managingEditor>acdlm88.8@proton.me</managingEditor>
<webMaster>acdlm88.8@proton.me</webMaster>

  <language>fr-fr</language>

  <itunes:category text="Society &amp; Culture">
    <itunes:category text="Arts"/>
  </itunes:category>
  
  <itunes:explicit>yes</itunes:explicit>
'''
    
    for ep in episodes:
        rss += f'''        <item>
      <title>{ep['title']}</title>
      <link>{ep['link']}</link>
      <guid isPermaLink="false">{ep['guid']}</guid>
      <description>{ep['description']}</description>
      <pubDate>{ep['pubDate']}</pubDate>
      <enclosure url="{ep['mp3_url']}" type="audio/mpeg"/>
      <itunes:duration>{ep['duration']}</itunes:duration>
      <itunes:episode>{ep['episode']}</itunes:episode>
      <itunes:season>{ep['season']}</itunes:season>
      <itunes:episodeType>full</itunes:episodeType>
      <itunes:image href="{ep['image']}"/>
    </item>
'''
    
    rss += '''  </channel>
</rss>'''
    
    return rss

In [37]:
with open('feed.xml','w') as f:
    f.write(generer_rss(episodes))

In [ ]:
s=slugs[0]
slug_response = requests.get(base_url+s)
slug_soup = BeautifulSoup(slug_response.content, 'html.parser')

In [28]:
slug_soup

<!DOCTYPE html>

<html lang="fr-FR">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="Radio Campus Besançon occupe les ondes 102.4 FM depuis 1997. Membre actif du réseau Radio Campus France, elle lie actualité culturelle et programmation musicale actuelle." name="description"/>
<link href="https://campusbelfortmontbeliard.fr/wp-content/themes/radiocampus-bm/assets/images/favicon/apple-touch-icon.png" rel="icon" type="image/png"/>
<meta content="index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1" name="robots"/>
<!-- This site is optimized with the Yoast SEO plugin v27.0 - https://yoast.com/product/yoast-seo-wordpress/ -->
<title>ACDLM S1-Ep8 TCG et Jeux de cartes | 24-03-2025 - Radio Campus Belfort Montbéliard</title>
<link href="https://campusbelfortmontbeliard.fr/replay/acdlm-s1-8-tcg-et-jeux-de-cartes-24-03-2025/" rel="canonical">
<meta content="fr_FR" property="og:locale"/>
<meta content="a